In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [2]:
train['TotalSpend'] = (
                        train['RoomService'].fillna(0) +
                        train['FoodCourt'].fillna(0) +
                        train['ShoppingMall'].fillna(0) +
                        train['Spa'].fillna(0) +
                        train['VRDeck'].fillna(0))

train[['Deck', 'Cabin_No','Side']] = train['Cabin'].str.split('/',expand=True)
train[['GroupID', 'MemberID']] = train['PassengerId'].str.split('_', expand=True)

group_counts = train['GroupID'].value_counts()
train["Group_Size"] = train["GroupID"].map(group_counts)

In [3]:
mask_spending = (train['CryoSleep'].isna()) & (train['TotalSpend'] > 0)
train.loc[mask_spending, 'CryoSleep'] = False

spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_cols:
    train.loc[(train['CryoSleep'] == True) &(train[col].isna()), col] = 0

cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
for col in cat_cols:
    if col in train.columns:
        train[col] = train[col].fillna(train[col].mode()[0])

num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    if col in train.columns:
        train[col] = train[col].fillna(train[col].median())

print("Remaining missing values:")
print(train.isna().sum().sum())

train.info()

Remaining missing values:
598
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8693 non-null   object 
 2   CryoSleep     8693 non-null   bool   
 3   Cabin         8494 non-null   object 
 4   Destination   8693 non-null   object 
 5   Age           8693 non-null   float64
 6   VIP           8693 non-null   bool   
 7   RoomService   8693 non-null   float64
 8   FoodCourt     8693 non-null   float64
 9   ShoppingMall  8693 non-null   float64
 10  Spa           8693 non-null   float64
 11  VRDeck        8693 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
 14  TotalSpend    8693 non-null   float64
 15  Deck          8693 non-null   object 
 16  Cabin_No      8494 non-null   object 
 17  Side          8693 non-null   object 
 18

C:\Users\baibh\AppData\Local\Temp\ipykernel_35964\3747930201.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train[col] = train[col].fillna(train[col].mode()[0])


In [ ]:
def impute_home_planet(row):
    # If HomePlanet is already known, keep it
    if not pd.isna(row['HomePlanet']):
        return row['HomePlanet']
    
    # If HomePlanet is missing, look at the Deck
    deck = row['Deck']
    
    # Rule 1: The 100% Matches
    if deck == 'G':
        return 'Earth'
    elif deck in ['A', 'B', 'C', 'T']:
        return 'Europa'
    
    # Rule 2: The "Best Guess" based on your Heatmap counts
    # For mixed decks, we pick the highest count from your image
    elif deck == 'D':
        return 'Mars'   # Mars (282) > Europa (186)
    elif deck == 'F':
        return 'Earth'  # Earth (1614) > Mars (1110)
    elif deck == 'E':
        return 'Earth'  # Earth (395) > Mars (330) > Europa (128)
    
    # Fallback (if Deck is also NaN)
    return pd.NA

# Apply the function
# Make sure you have split the Cabin column first!
train['HomePlanet'] = train.apply(impute_home_planet, axis=1)

# Fill any remaining NAs (where Deck was also missing) with the global mode
train['HomePlanet'] = train['HomePlanet'].fillna(train['HomePlanet'].mode()[0])

print("Smart Imputation Complete.")

In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
import lightgbm as lgb

features_to_drop = ['Name', 'PassengerId', 'Cabin', 'GroupID', 'MemberID']
model_df = train.drop(columns=[c for c in features_to_drop if c in train.columns], errors='ignore').copy()

model_df['Transported'] = model_df['Transported'].astype(int)

cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']

le = LabelEncoder()

for col in cat_cols:
    if col in model_df.columns:

        model_df[col] = model_df[col].astype(str)
        model_df[col] = le.fit_transform(model_df[col])

if 'Cabin_No' in model_df.columns:
    model_df['Cabin_No'] = pd.to_numeric(model_df['Cabin_No'], errors='coerce').fillna(-1)

X = model_df.drop('Transported', axis=1)
y = model_df['Transported']

print("Features used: ", list(X.columns))
print(X.info())

Features used:  ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend', 'Deck', 'Cabin_No', 'Side', 'Group_Size']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    8693 non-null   int32  
 1   CryoSleep     8693 non-null   int32  
 2   Destination   8693 non-null   int32  
 3   Age           8693 non-null   float64
 4   VIP           8693 non-null   int32  
 5   RoomService   8693 non-null   float64
 6   FoodCourt     8693 non-null   float64
 7   ShoppingMall  8693 non-null   float64
 8   Spa           8693 non-null   float64
 9   VRDeck        8693 non-null   float64
 10  TotalSpend    8693 non-null   float64
 11  Deck          8693 non-null   int32  
 12  Cabin_No      8693 non-null   float64
 13  Side          8693 non-null   int32  
 14  Group_Size

In [5]:
import warnings

warnings.filterwarnings('ignore')

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    eval_metrics='logloss',
    use_label_encoder=False,
    random_state=100
)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)

xgb_scores = cross_val_score(xgb_model, X, y, cv=kf, scoring='accuracy')

print(f"XGBoost acc per fold {xgb_scores}")
print(f'XGBoost mean acc.: {xgb_scores.mean():.4f}')

XGBoost acc per fold [0.80046003 0.80621047 0.81943646 0.80552359 0.80609896]
XGBoost mean acc.: 0.8075


In [6]:
lgbm_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    random_state=100,
    verbose=-1
)

lgbm_scores = cross_val_score(lgbm_model, X, y, cv=kf, scoring='accuracy')

print(f"LightGBM acc per fold: {lgbm_scores}")
print(f"LGBM Mean Acc. : {lgbm_scores.mean():.4f}")

LightGBM acc per fold: [0.80218516 0.81426107 0.82173663 0.81012658 0.80782509]
LGBM Mean Acc. : 0.8112


In [7]:
from sklearn.ensemble import VotingClassifier

voting_model = VotingClassifier(
    estimators=[('xgb', xgb_model), ('lgbm', lgbm_model)], voting='soft')

voting_scores = cross_val_score(voting_model, X, y, cv=kf, scoring='accuracy')

print(f"Ensemble mean acc. : {voting_scores.mean():.4f}")

Ensemble mean acc. : 0.8093


In [8]:
import optuna


def objective(trial):

    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 100,

        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 15),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_Samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }


    model = lgb.LGBMClassifier(**params)

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)
    scores = cross_val_score(model, X, y, cv = kf, scoring='accuracy')

    return scores.mean()

study = optuna.create_study(direction='maximize')

print("Hyper parameter sweep ...")
study.optimize(objective, n_trials=50)

print("-" * 30)
print(f"Best accuracy: {study.best_value:.4f}")
print("Best parameters")
print(study.best_params)

[I 2026-01-24 17:59:19,745] A new study created in memory with name: no-name-53ed8d38-3d5a-47f8-872b-ace11e63b947


Hyper parameter sweep ...


[I 2026-01-24 17:59:22,420] Trial 0 finished with value: 0.8076606464702344 and parameters: {'n_estimators': 220, 'learning_rate': 0.01245654408132982, 'num_leaves': 84, 'max_depth': 10, 'reg_alpha': 0.6027728794211704, 'reg_lambda': 0.573639045747209, 'min_child_Samples': 22, 'subsample': 0.8206015536350171, 'colsample_bytree': 0.9174341904550863}. Best is trial 0 with value: 0.8076606464702344.
[I 2026-01-24 17:59:23,604] Trial 1 finished with value: 0.8055906897275064 and parameters: {'n_estimators': 578, 'learning_rate': 0.05935102765226871, 'num_leaves': 82, 'max_depth': 3, 'reg_alpha': 0.0009548054769986268, 'reg_lambda': 0.009117267273228338, 'min_child_Samples': 55, 'subsample': 0.8478770426537438, 'colsample_bytree': 0.7488738910902205}. Best is trial 0 with value: 0.8076606464702344.
[I 2026-01-24 17:59:28,204] Trial 2 finished with value: 0.8106515324667762 and parameters: {'n_estimators': 435, 'learning_rate': 0.01771069837325629, 'num_leaves': 93, 'max_depth': 12, 'reg_alp

------------------------------
Best accuracy: 0.8164
Best parameters
{'n_estimators': 783, 'learning_rate': 0.007433727090951476, 'num_leaves': 52, 'max_depth': 14, 'reg_alpha': 0.4479937583433854, 'reg_lambda': 1.3372805712109775e-06, 'min_child_Samples': 5, 'subsample': 0.5602456728217275, 'colsample_bytree': 0.9051682503760634}


In [9]:
from optuna.visualization import plot_optimization_history, plot_param_importances

plot_optimization_history(study).show()

plot_param_importances(study).show()

In [13]:
best_params = study.best_params
best_lgbm = lgb.LGBMClassifier(**best_params, random_state=42)

best_lgbm.fit(X, y)


LGBMClassifier(colsample_bytree=0.9051682503760634,
               learning_rate=0.007433727090951476, max_depth=14,
               min_child_Samples=5, n_estimators=783, num_leaves=52,
               random_state=42, reg_alpha=0.4479937583433854,
               reg_lambda=1.3372805712109775e-06, subsample=0.5602456728217275)

In [18]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='Accuracy',
    verbose=0, # Silent training
    random_state=42
)

cat_scores = cross_val_score(cat_model, X, y, cv=kf, scoring='accuracy')

print(f"CatBoost acc per fold: {cat_scores}")
print(f"CatBoost Mean Acc. : {cat_scores.mean():.4f}")

CatBoost acc per fold: [0.80448534 0.80908568 0.81828637 0.81300345 0.81127733]
CatBoost Mean Acc. : 0.8112


In [19]:
# data for submission

test = pd.read_csv("test.csv")
submission_id = test['PassengerId'].copy()

def preprocess_data(df):
    df = df.copy()

    df[['Deck', 'Cabin_No', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df[['GroupID', 'MemberID']] = df['PassengerId'].str.split('_', expand=True)

    group_counts = df['GroupID'].value_counts()
    df["Group_Size"] = df["GroupID"].map(group_counts)

    df['TotalSpend'] = (
                        df['RoomService'].fillna(0) +
                        df['FoodCourt'].fillna(0) +
                        df['ShoppingMall'].fillna(0) +
                        df['Spa'].fillna(0) +
                        df['VRDeck'].fillna(0))

    mask_spending = (df['CryoSleep'].isna()) & (df['TotalSpend'] > 0)
    df.loc[mask_spending, 'CryoSleep'] = False

    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in spend_cols:
        df.loc[(df['CryoSleep'] == True) &(df[col].isna()), col] = 0

    cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in num_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    
    cols_drop = ['Name', 'PassengerId', 'Cabin', 'GroupID', 'MemberID']
    df = df.drop(columns=[c for c in cols_drop if c in df.columns], errors='ignore')

    cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
    le = LabelEncoder()
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)
            df[col] = le.fit_transform(df[col])

    if 'Cabin_No' in df.columns:
        df['Cabin_No'] = pd.to_numeric(df['Cabin_No'], errors='coerce').fillna(-1)

    return df

X_test = preprocess_data(test)

X_test.info()
            

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4277 entries, 0 to 4276
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    4277 non-null   int32  
 1   CryoSleep     4277 non-null   int32  
 2   Destination   4277 non-null   int32  
 3   Age           4277 non-null   float64
 4   VIP           4277 non-null   int32  
 5   RoomService   4277 non-null   float64
 6   FoodCourt     4277 non-null   float64
 7   ShoppingMall  4277 non-null   float64
 8   Spa           4277 non-null   float64
 9   VRDeck        4277 non-null   float64
 10  Deck          4277 non-null   int32  
 11  Cabin_No      4277 non-null   float64
 12  Side          4277 non-null   int32  
 13  Group_Size    4277 non-null   int64  
 14  TotalSpend    4277 non-null   float64
dtypes: float64(8), int32(6), int64(1)
memory usage: 401.1 KB


In [20]:
# training the test dataset for submission

X_test = X_test.reindex(columns=X.columns, fill_value=0)

cat_model.fit(X, y)

y_pred = cat_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': submission_id,
    'Transported': y_pred.astype(bool)
})

submission.to_csv('submission.csv', index=False)
print("Submission saved")

Submission saved


In [21]:
sample = pd.read_csv('sample_submission.csv')
print(sample.shape)
print(submission.shape)

(4277, 2)
(4277, 2)
